# Assessment Analysis Dashboard  
### Central Interface for Coursework Tools  
**Student ID:** F416013  

This notebook provides a unified dashboard for launching the various assessment analysis tools developed for the module.  
Use the menu on the left to run preprocessing, view test results, explore question-level performance, or identify underperforming students.

import os
import time
import subprocess
from datetime import datetime

import ipywidgets as widgets
from IPython.display import display, clear_output

In [1]:
import os
import time
import subprocess
from datetime import datetime

import ipywidgets as widgets
from IPython.display import display, clear_output

In [2]:
# ============================================================
# DASHBOARD ENGINE
# (Theme, logging, static content, and actions)
# ============================================================

# Theme definitions as module-level constants (removes internal if/else)
LIGHT_THEME = {
    "bg": "#FFFFFF",
    "panel": "#F5F5F5",
    "text": "#000000",
    "muted": "#555555",
    "border": "#C8C8C8",
    "header": "#005A9E",
    "success_bg": "#D4EDDA",
    "success_text": "#155724",
    "error_bg": "#F8D7DA",
    "error_text": "#721C24",
}

DARK_THEME = {
    "bg": "#2B2B2B",
    "panel": "#3A3A3A",
    "text": "#F2F2F2",
    "muted": "#CFCFCF",
    "border": "#5A5A5A",
    "header": "#4CC2FF",
    "success_bg": "#27472F",
    "success_text": "#D4EDDA",
    "error_bg": "#5A1F25",
    "error_text": "#F8D7DA",
}


class DashboardEngine:
    """
    Provides the core functionality behind the dashboard, including
    theme management, logging, static interface elements, and the
    callback actions for launching external tools. The aim is to
    keep the main application class focused on layout and user
    interaction.
    """

    def __init__(self):
        self.dark = False
        self._theme = LIGHT_THEME

        # Logger widget
        self.log_widget = widgets.Textarea(
            value="",
            disabled=True,
            layout=widgets.Layout(
                width="100%",
                height="200px",
                font_family="monospace",
                font_size="11px",
            ),
        )

        # Build static UI components
        self.header = None
        self.about = None
        self.footer = None
        self.refresh_statics()

    # ---------------------- Theme handling ----------------------

    def set_dark(self, value: bool):
        """Enable or disable the dark theme and refresh static content."""
        self.dark = bool(value)
        self._theme = DARK_THEME if self.dark else LIGHT_THEME
        self.refresh_statics()

    def colours(self):
        """Return the current colour palette."""
        return self._theme

    # ---------------------- Logging ----------------------

    def log(self, msg: str):
        """Append a timestamped message to the log panel."""
        t = datetime.now().strftime("%H:%M:%S")
        line = f"[{t}] {msg}"
        if self.log_widget.value:
            self.log_widget.value += "\n" + line
            return
        self.log_widget.value = line

    # ---------------------- Static UI ----------------------

    def refresh_statics(self):
        """Rebuild header, about panel, and footer for the current theme."""
        c = self.colours()

        self.header = widgets.HTML(
            f"""
<h1 style='color:{c["header"]}; margin-bottom:0; font-family:Arial;'>
    Assessment Analysis Dashboard
</h1>
<p style='color:{c["text"]}; margin-top:5px; font-family:Arial;'>
    A central interface for launching assessment analysis tools.
</p>
"""
        )

        about_html = widgets.HTML(
            f"""
<div style='color:{c["text"]}; font-family:Arial; font-size:14px;'>
    <p><b>Module:</b> 25COP504</p>
    <p><b>Purpose:</b> A unified environment for exploring and analysing student assessment data.</p>
    <ul>
        <li>Data Preprocessing</li>
        <li>Test Results Viewer</li>
        <li>Question-Level Performance</li>
        <li>Underperforming Students Tool</li>
    </ul>
</div>
"""
        )

        acc = widgets.Accordion(children=[about_html])
        acc.set_title(0, "About this project")
        self.about = acc

        self.footer = widgets.HTML(
            f"""
<p style='color:{c["muted"]}; font-size:11px; text-align:right; font-family:Arial;'>
    Version 1.0.0 — Last updated: 29 Dec 2025
</p>
"""
        )

    def banner_success(self, msg):
        """Return a green success banner."""
        c = self.colours()
        return widgets.HTML(
            f"""
<div style='background:{c["success_bg"]}; color:{c["success_text"]};
            padding:10px; border-left:5px solid #2E8540;'>
    {msg}
</div>
"""
        )

    def banner_error(self, msg):
        """Return a red error banner."""
        c = self.colours()
        return widgets.HTML(
            f"""
<div style='background:{c["error_bg"]}; color:{c["error_text"]};
            padding:10px; border-left:5px solid #B10E1E;'>
    {msg}
</div>
"""
        )

    # ---------------------- Actions ----------------------

    def _show_banner(self, output, banner_widget):
        """Utility to display a banner in the output area."""
        with output:
            clear_output()
            display(banner_widget)

    def launch_script(self, script_name, output):
        """Launch an external Python script in a separate process."""
        try:
            subprocess.Popen(["python3", script_name])
        except Exception as e:
            self._show_banner(output, self.banner_error(str(e)))
            self.log(str(e))
            return

        self._show_banner(output, self.banner_success(f"{script_name} launched."))
        self.log(f"Launched {script_name}.")

    def run_preprocessing(self, output, pbar, plabel):
        """Launch preprocessing and simulate progress updates."""
        with output:
            clear_output()

        self.log("Starting preprocessing...")
        pbar.value = 0
        pbar.bar_style = ""
        plabel.value = ""

        try:
            subprocess.Popen(["python3", "CWPreprocessing.py"])
        except Exception as e:
            self._show_banner(output, self.banner_error(str(e)))
            self.log(str(e))
            return

        steps = [
            ("Loading CSV files...", 20),
            ("Cleaning data...", 45),
            ("Calculating scaled proportions...", 70),
            ("Updating database...", 90),
            ("Finalising...", 100),
        ]

        for text, val in steps:
            c = self.colours()
            plabel.value = f"<span style='color:{c['text']};'>{text}</span>"
            pbar.value = val
            self.log(text)
            time.sleep(0.6)

        pbar.bar_style = "success"
        self._show_banner(output, self.banner_success("Data preprocessing completed."))
        self.log("Preprocessing complete.")

    def show_help(self, output):
        """Display a brief help section."""
        with output:
            clear_output()
            c = self.colours()
            display(
                widgets.HTML(
                    f"""
<h3 style='color:{c["text"]};'>Help</h3>
<ul style='color:{c["text"]};'>
    <li>Use the menu on the left to launch tools.</li>
    <li>Run preprocessing whenever CSV files change.</li>
    <li>Each tool opens in a separate window.</li>
    <li>Check the system log for errors or status updates.</li>
</ul>
"""
                )
            )

    def reset_database(self, output):
        """Delete the main database file if it exists."""
        with output:
            clear_output()

        db = "Resultdatabase.db"

        if not os.path.exists(db):
            display(self.banner_error("Resultdatabase.db does not exist."))
            self.log("Database not found.")
            return

        try:
            os.remove(db)
        except Exception as e:
            display(self.banner_error(str(e)))
            self.log(str(e))
            return

        display(self.banner_success("Resultdatabase.db deleted."))
        self.log("Database deleted.")

In [3]:
# ============================================================
# DASHBOARD APPLICATION (UI LAYER)
# ============================================================

class DashboardApp:
    """
    Constructs the dashboard layout, wires button callbacks, and
    applies the selected theme. The engine handles all underlying
    logic, allowing this class to focus on presentation.
    """

    def __init__(self):
        self.engine = DashboardEngine()

        # Output and progress widgets
        self.output = widgets.Output(
            layout=widgets.Layout(
                border="1px solid #ccc",
                padding="10px",
            )
        )

        self.pbar = widgets.IntProgress(
            value=0,
            min=0,
            max=100,
            layout=widgets.Layout(width="100%"),
        )

        self.plabel = widgets.HTML("")

        # Buttons
        self.btn_pre = widgets.Button(
            description="Data Preprocessing",
            layout=widgets.Layout(width="230px"),
        )
        self.btn_test = widgets.Button(
            description="Test Results Viewer",
            layout=widgets.Layout(width="230px"),
        )
        self.btn_perf = widgets.Button(
            description="Student Performance",
            layout=widgets.Layout(width="230px"),
        )
        self.btn_under = widgets.Button(
            description="Underperforming Students",
            layout=widgets.Layout(width="230px"),
        )
        self.btn_help = widgets.Button(
            description="Help / Instructions",
            layout=widgets.Layout(width="230px"),
        )
        self.btn_reset = widgets.Button(
            description="Reset Database",
            layout=widgets.Layout(width="230px"),
        )
        self.btn_log = widgets.Button(
            description="View Log",
            layout=widgets.Layout(width="230px"),
        )

        self.dark_toggle = widgets.Checkbox(value=False, description="Dark mode")

        self._build_layout()
        self._wire_callbacks()
        self._apply_theme()
        self._initial_output()

    # ------------------ Layout ------------------
                
    def _build_layout(self):
        """Construct the left-hand menu and main content area."""
        left = widgets.VBox(
            [
                widgets.HTML("<h3>Tools</h3>"),
                self.btn_pre,
                self.btn_test,
                self.btn_perf,
                self.btn_under,
                self.btn_help,
                self.btn_reset,
                self.btn_log,
                self.dark_toggle,
            ],
            layout=widgets.Layout(
                width="260px",
                padding="10px",
                border="1px solid #C8C8C8",
            ),
        )

        progress_box = widgets.VBox(
            [
                widgets.HTML("<b>Preprocessing progress</b>"),
                self.pbar,
                self.plabel,
            ]
        )

        right = widgets.VBox(
            [
                self.engine.header,
                self.output,
                progress_box,
                widgets.HTML("<b>System log</b>"),
                self.engine.log_widget,
                self.engine.about,
                self.engine.footer,
            ],
            layout=widgets.Layout(
                flex="1",
                padding="10px",
                border="1px solid #C8C8C8",
            ),
        )

        self.container = widgets.HBox(
            [left, right],
            layout=widgets.Layout(width="100%"),
        )

    # ------------------ Callbacks ------------------

    def _wire_callbacks(self):
        """Attach button callbacks and theme toggle behaviour."""
        # Use small helper lambdas; logic lives in engine
        self.btn_pre.on_click(
            lambda _: self.engine.run_preprocessing(self.output, self.pbar, self.plabel)
        )
        self.btn_test.on_click(
            lambda _: self.engine.launch_script("testResults.py", self.output)
        )
        self.btn_perf.on_click(
            lambda _: self.engine.launch_script("studentPerformance.py", self.output)
        )
        self.btn_under.on_click(
            lambda _: self.engine.launch_script("underperformingStudent.py", self.output)
        )
        self.btn_help.on_click(lambda _: self.engine.show_help(self.output))
        self.btn_reset.on_click(lambda _: self.engine.reset_database(self.output))
        self.btn_log.on_click(lambda _: self._show_log())

        # Simplify theme toggle with early return
        def on_toggle(change):
            if change.get("name") != "value":
                return
            self.engine.set_dark(change.get("new", False))
            self._apply_theme()

        self.dark_toggle.observe(on_toggle)

    def _show_log(self):
        """Display the system log."""
        with self.output:
            clear_output()
            display(self.engine.log_widget)

    # ------------------ Theme ------------------

    def _apply_theme(self):
        """Apply the selected theme to the dashboard."""
        c = self.engine.colours()
        self.container.layout.background_color = c["bg"]
        self.output.layout.background_color = c["panel"]
        self.output.layout.border = f"1px solid {c['border']}"

        self.engine.log_widget.style = {
            "text_color": c["text"],
            "background": c["panel"],
        }
        self.engine.log_widget.layout.border = f"1px solid {c['border']}"

        # Rebuild static HTML to reflect the current theme
        self.engine.refresh_statics()

    # ------------------ Initial Output ------------------

    def _initial_output(self):
        """Initialise the output panel and log startup."""
        self.engine.log("Dashboard initialised.")
        with self.output:
            clear_output()

    # ------------------ Run ------------------

    def run(self):
        """Display the dashboard."""
        display(self.container)

In [4]:
DashboardApp().run()